In [3]:
# Import Necessary Libraries
import pandas as pd

In [5]:
# Load the transformed AIS data
file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df = pd.read_csv(file_path)

In [6]:
# Initialize a list to store aggregated voyage data
voyage_data = []

In [7]:
# Group by 'mmsi' and 'Voyage_ID' to process each voyage
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Sort the group by timestamp to ensure chronological order
    group = group.sort_values('timestamp').reset_index(drop=True)
    
    # Extract first and last row of the voyage
    first_row = group.iloc[0]
    last_row = group.iloc[-1]
    
    # Ensure DepartureTime is earlier than ArrivalTime
    departure_time = first_row['timestamp']
    arrival_time = last_row['timestamp']
    if pd.to_datetime(departure_time) > pd.to_datetime(arrival_time):
        # Swap departure and arrival times if necessary
        departure_time, arrival_time = arrival_time, departure_time
    
    # Create a dictionary for the aggregated voyage data
    voyage_dict = {
        'mmsi': mmsi,
        'Voyage_ID': voyage_id,
        'DepartureTime': departure_time,
        'ArrivalTime': arrival_time,
        'LATd': first_row['lat'],
        'LONd': first_row['lon'],
        'LATa': last_row['lat'],
        'LONa': last_row['lon'],
        'breadth': first_row['breadth'],
        'Voyage_Distance_Km': first_row['Voyage_Distance'],
        'vessel_type': first_row['vessel_type'],
        'vessel_max_speed_kmph': group['speedKMpH'].max(),  # Maximum value of speedKM
        'vessel_min_speed_kmph': group['speedKMpH'].min(),  # Minimum value of speedKM
        'draft_km': first_row['draft'],
        'Start_Port': first_row['Port_Name'] if pd.notna(first_row['Port_Name']) else '',
        'End_Port': last_row['Port_Name'] if pd.notna(last_row['Port_Name']) else '',
        'Weighted_Avg_Speed_kmph': first_row['Weighted_Avg_Speed'],
        'Avg_Course': first_row['Avg_Course'],
        'AvgSpeedkmph': first_row['AvgSpeedkmph'],
        'Std_SpeedKMpH': first_row['Std_SpeedKMpH'],
        'Std_Course': first_row['Std_Course'],
        'Heading_Change_Sum': first_row['Heading_Change_Sum']
    }
    
    # Append the voyage data to the list
    voyage_data.append(voyage_dict)

In [8]:
# Create a new DataFrame from the aggregated voyage data
aggregated_voyage_df = pd.DataFrame(voyage_data)

# Save the aggregated voyage data to a new file
output_file_path = "D:\\AIS Project\\datasets\\Aggregated_Voyage_Data.csv"
aggregated_voyage_df.to_csv(output_file_path, index=False)

print(f"Aggregated voyage data saved to {output_file_path}")

Aggregated voyage data saved to D:\AIS Project\datasets\Aggregated_Voyage_Data.csv


In [9]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Aggregated_Voyage_Data.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Count the total number of rows
total_rows = len(aggregated_voyage_df)
print(f"Total number of rows: {total_rows}")

# Count the number of unique mmsi (vessels)
unique_mmsi_count = aggregated_voyage_df['mmsi'].nunique()
print(f"Number of unique mmsi (vessels): {unique_mmsi_count}")

# Calculate the maximum and minimum number of voyages for each mmsi
voyage_counts = aggregated_voyage_df.groupby('mmsi')['Voyage_ID'].nunique()
max_voyages = voyage_counts.max()
min_voyages = voyage_counts.min()

print(f"Maximum number of voyages by a single vessel: {max_voyages}")
print(f"Minimum number of voyages by a single vessel: {min_voyages}")

Total number of rows: 48936
Number of unique mmsi (vessels): 1750
Maximum number of voyages by a single vessel: 618
Minimum number of voyages by a single vessel: 1


In [10]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Aggregated_Voyage_Data.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Convert 'breadth' from meters to kilometers and rename the column to 'breadthKM'
aggregated_voyage_df['breadthKM'] = aggregated_voyage_df['breadth'] / 1000
aggregated_voyage_df.drop(columns=['breadth'], inplace=True)  # Drop the original 'breadth' column

# Exclude voyages with a maximum speed greater than 42.596 km/h
aggregated_voyage_df = aggregated_voyage_df[aggregated_voyage_df['vessel_max_speed_kmph'] <= 42.596]

# Exclude voyages with a voyage distance less than 185.2 km
aggregated_voyage_df = aggregated_voyage_df[aggregated_voyage_df['Voyage_Distance_Km'] >= 555.6]

# Exclude voyages with the same starting and ending ports/coordinates (rounded to two decimal place)
aggregated_voyage_df = aggregated_voyage_df[
    ~(
        (aggregated_voyage_df['LATd'].round(1) == aggregated_voyage_df['LATa'].round(1)) &
        (aggregated_voyage_df['LONd'].round(1) == aggregated_voyage_df['LONa'].round(1))
    )
]

# Drop unnecessary columns
if 'Start_Port' in aggregated_voyage_df.columns:
    aggregated_voyage_df.drop(columns=['Start_Port'], inplace=True)
if 'End_Port' in aggregated_voyage_df.columns:
    aggregated_voyage_df.drop(columns=['End_Port'], inplace=True)
if 'Voyage_ID' in aggregated_voyage_df.columns:
    aggregated_voyage_df.drop(columns=['Voyage_ID'], inplace=True)

# Update the 'vessel_type' column based on the specified mappings
vessel_type_mapping = {
    'T': 80,
    'RORO': 90,
    'PAS': 60,
    'GC': 70,
    'CONT': 71
}
aggregated_voyage_df['vessel_type'] = aggregated_voyage_df['vessel_type'].map(vessel_type_mapping)

# Save the updated dataframe to the new file
output_file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data.csv"
aggregated_voyage_df.to_csv(output_file_path, index=False)

print(f"Filtered and updated aggregated voyage data saved to {output_file_path}")

Filtered and updated aggregated voyage data saved to D:\AIS Project\datasets\Feature_Engineered_Data.csv


In [11]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Count the total number of rows
total_rows = len(aggregated_voyage_df)
print(f"Total number of rows: {total_rows}")

# Count the number of unique mmsi (vessels)
unique_mmsi_count = aggregated_voyage_df['mmsi'].nunique()
print(f"Number of unique mmsi (vessels): {unique_mmsi_count}")

Total number of rows: 19827
Number of unique mmsi (vessels): 1504


In [14]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Define the bins and labels for segregation
bins = [0, 555.6, 926, 1852, float('inf')] # 300 nm = 555.6 km, 500 nm = 926 km, 1000 nm = 1852 km
labels = ['very small', 'small', 'medium', 'large'] # very small < 300, 300 <= small < 500, 500 <= medium <1000, large <= 1000 (in nm)

# Create a new column 'Voyage_Distance_Category' based on the bins and labels
aggregated_voyage_df['Voyage_Distance_Category'] = pd.cut(
    aggregated_voyage_df['Voyage_Distance_Km'], 
    bins=bins, 
    labels=labels, 
    right=False
)

# Save the updated dataframe to the new file
output_file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data_with_Category.csv"
aggregated_voyage_df.to_csv(output_file_path, index=False)

print(f"Updated aggregated voyage data with distance category saved to {output_file_path}")

Updated aggregated voyage data with distance category saved to D:\AIS Project\datasets\Feature_Engineered_Data_with_Category.csv


In [17]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data_with_Category.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Convert 'ArrivalTime' and 'DepartureTime' to datetime objects
aggregated_voyage_df['ArrivalTime'] = pd.to_datetime(aggregated_voyage_df['ArrivalTime'])
aggregated_voyage_df['DepartureTime'] = pd.to_datetime(aggregated_voyage_df['DepartureTime'])

# Calculate the difference in hours
aggregated_voyage_df['Duration_Hours'] = (
    (aggregated_voyage_df['ArrivalTime'] - aggregated_voyage_df['DepartureTime']).dt.total_seconds() / 3600
)

# Save the updated dataframe to the new file
output_file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data_with_Duration.csv"
aggregated_voyage_df.to_csv(output_file_path, index=False)

print(f"Updated aggregated voyage data with duration in hours saved to {output_file_path}")

Updated aggregated voyage data with duration in hours saved to D:\AIS Project\datasets\Feature_Engineered_Data_with_Duration.csv


In [27]:
# Load the data
file_path = "D:\\AIS Project\\datasets\\Feature_Engineered_Data_with_Duration.csv"
aggregated_voyage_df = pd.read_csv(file_path)

# Convert 'ArrivalTime' and 'DepartureTime' to datetime (if not already done)
aggregated_voyage_df['ArrivalTime'] = pd.to_datetime(aggregated_voyage_df['ArrivalTime'])
aggregated_voyage_df['DepartureTime'] = pd.to_datetime(aggregated_voyage_df['DepartureTime'])

# Calculate voyage speed (Km/h)
aggregated_voyage_df['Calculated_Speed_Kmph'] = (
    aggregated_voyage_df['Voyage_Distance_Km'] / aggregated_voyage_df['Duration_Hours']
)

# Define thresholds
min_duration_hours = 13.0435  # Minimum duration for a voyage = (min of) distance_km / (max of)vessel_max_speed_kmph; (in hours)
min_distance_km = 555.6  # Allowable minimum distance is 555.6 km or 300 nm

# Identify discrepancies
discrepancies = (
    (aggregated_voyage_df['Duration_Hours'] < min_duration_hours) |  # Voyage duration too short
    (aggregated_voyage_df['Voyage_Distance_Km'] < min_distance_km)   # Voyage distance too short
)

# Filter out rows with discrepancies and create a copy
cleaned_df = aggregated_voyage_df[~discrepancies].copy()

# Drop the temporary 'Calculated_Speed_Kmph' column
cleaned_df.drop(columns=['Calculated_Speed_Kmph'], inplace=True)

# Save the cleaned data to a new file
output_file_path = "D:\\AIS Project\\datasets\\Cleaned_Voyage_Data.csv"
cleaned_df.to_csv(output_file_path, index=False)

print(f"Cleaned voyage data saved to {output_file_path}")
print(f"Number of rows removed: {len(aggregated_voyage_df) - len(cleaned_df)}")

Cleaned voyage data saved to D:\AIS Project\datasets\Cleaned_Voyage_Data.csv
Number of rows removed: 34


In [29]:
# Load the aggregated voyage data
file_path = "D:\\AIS Project\\datasets\\Cleaned_Voyage_Data.csv"
cleaned_df = pd.read_csv(file_path)

# Count the total number of rows
total_rows = len(cleaned_df)
print(f"Total number of rows: {total_rows}")

Total number of rows: 19793


In [33]:
# Count the occurrences of "small", "medium", and "large" in the "Voyage_Distance_Category" column
category_counts = cleaned_df['Voyage_Distance_Category'].value_counts()

# Calculate the percentage distribution
category_percentages = (category_counts / len(cleaned_df)) * 100

# Display the counts and percentages
print("Counts of Voyage Distance Categories:")
print(category_counts)
print("\nPercentage Distribution of Voyage Distance Categories:")
print(category_percentages)

# Calculate max, average, and min of "Voyage_Distance_Km" and "Duration_Hours"
voyage_distance_stats = {
    'Max_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].max(),
    'Avg_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].mean(),
    'Min_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].min()
}

duration_hours_stats = {
    'Max_Duration_Hours': cleaned_df['Duration_Hours'].max(),
    'Avg_Duration_Hours': cleaned_df['Duration_Hours'].mean(),
    'Min_Duration_Hours': cleaned_df['Duration_Hours'].min()
}

# Display the statistics
print("\nStatistics for Voyage_Distance_Km:")
print(voyage_distance_stats)
print("\nStatistics for Duration_Hours:")
print(duration_hours_stats)

Counts of Voyage Distance Categories:
Voyage_Distance_Category
medium    10167
small      8228
large      1398
Name: count, dtype: int64

Percentage Distribution of Voyage Distance Categories:
Voyage_Distance_Category
medium    51.366645
small     41.570252
large      7.063103
Name: count, dtype: float64

Statistics for Voyage_Distance_Km:
{'Max_Voyage_Distance_Km': 6124.581000362377, 'Avg_Voyage_Distance_Km': 1128.5084657385794, 'Min_Voyage_Distance_Km': 555.721838608369}

Statistics for Duration_Hours:
{'Max_Duration_Hours': 8334.465555555556, 'Avg_Duration_Hours': 220.69729510152294, 'Min_Duration_Hours': 13.080277777777775}


In [43]:
# Define thresholds
max_duration_hours = 720  # 30 days in hours (30 * 24 = 720 hours)
min_speed_kmph = 1  # Minimum allowable speed (in km/h)
max_speed_kmph = 42.596  # Maximum allowable speed (in km/h)

# Calculate actual voyage speed
cleaned_df['Calculated_Speed_Kmph'] = (
    cleaned_df['Voyage_Distance_Km'] / cleaned_df['Duration_Hours']
)

# Identify abnormalities
abnormalities = (
    (cleaned_df['Duration_Hours'] > max_duration_hours) |  # Voyage duration too long
    (cleaned_df['Calculated_Speed_Kmph'] < min_speed_kmph) |  # Voyage speed too low
    (cleaned_df['Calculated_Speed_Kmph'] > max_speed_kmph)  # Voyage speed too high
)

# Filter out rows with abnormalities
cleaned_df = cleaned_df[~abnormalities].copy()

# Drop the temporary 'Calculated_Speed_Kmph' column
cleaned_df.drop(columns=['Calculated_Speed_Kmph'], inplace=True)

# Save the cleaned data to a new file with a custom name
output_file_path = "D:\\AIS Project\\datasets\\Final_Cleaned_Voyage_Data.csv" 
cleaned_df.to_csv(output_file_path, index=False)

print(f"Cleaned voyage data saved to {output_file_path}")
print(f"Number of rows removed: {len(aggregated_voyage_df) - len(cleaned_df)}")

Cleaned voyage data saved to D:\AIS Project\datasets\Final_Cleaned_Voyage_Data.csv
Number of rows removed: 1058


In [45]:
# Count the occurrences of "small", "medium", and "large" in the "Voyage_Distance_Category" column
category_counts = cleaned_df['Voyage_Distance_Category'].value_counts()

# Calculate the percentage distribution
category_percentages = (category_counts / len(cleaned_df)) * 100

# Display the counts and percentages
print("Counts of Voyage Distance Categories:")
print(category_counts)
print("\nPercentage Distribution of Voyage Distance Categories:")
print(category_percentages)

# Calculate max, average, and min of "Voyage_Distance_Km" and "Duration_Hours"
voyage_distance_stats = {
    'Max_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].max(),
    'Avg_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].mean(),
    'Min_Voyage_Distance_Km': cleaned_df['Voyage_Distance_Km'].min()
}

duration_hours_stats = {
    'Max_Duration_Hours': cleaned_df['Duration_Hours'].max(),
    'Avg_Duration_Hours': cleaned_df['Duration_Hours'].mean(),
    'Min_Duration_Hours': cleaned_df['Duration_Hours'].min()
}

# Display the statistics
print("\nStatistics for Voyage_Distance_Km:")
print(voyage_distance_stats)
print("\nStatistics for Duration_Hours:")
print(duration_hours_stats)

Counts of Voyage Distance Categories:
Voyage_Distance_Category
medium    9694
small     7995
large     1080
Name: count, dtype: int64

Percentage Distribution of Voyage Distance Categories:
Voyage_Distance_Category
medium    51.648996
small     42.596835
large      5.754169
Name: count, dtype: float64

Statistics for Voyage_Distance_Km:
{'Max_Voyage_Distance_Km': 6124.581000362377, 'Avg_Voyage_Distance_Km': 1101.9388591011748, 'Min_Voyage_Distance_Km': 555.721838608369}

Statistics for Duration_Hours:
{'Max_Duration_Hours': 719.4925, 'Avg_Duration_Hours': 86.8160444379325, 'Min_Duration_Hours': 15.552777777777775}


In [47]:
# Calculate the ratio of Voyage_Distance_Km to Duration_Hours (average speed in km/h)
cleaned_df['Average_Speed_Kmph'] = cleaned_df['Voyage_Distance_Km'] / cleaned_df['Duration_Hours']

# Calculate highest, average, and lowest values of the ratio
speed_stats = {
    'Highest_Speed_Kmph': cleaned_df['Average_Speed_Kmph'].max(),
    'Average_Speed_Kmph': cleaned_df['Average_Speed_Kmph'].mean(),
    'Lowest_Speed_Kmph': cleaned_df['Average_Speed_Kmph'].min()
}

# Display the statistics
print("Statistics for Voyage_Distance_Km / Duration_Hours (Average Speed in km/h):")
print(speed_stats)

Statistics for Voyage_Distance_Km / Duration_Hours (Average Speed in km/h):
{'Highest_Speed_Kmph': 42.409285497001, 'Average_Speed_Kmph': 19.56524217644599, 'Lowest_Speed_Kmph': 1.001210777843916}


In [49]:
# Calculate the average of the 'vessel_min_speed_kmph' column
average_min_speed = cleaned_df['vessel_min_speed_kmph'].mean()

# Display the average
print(f"Average of 'vessel_min_speed_kmph': {average_min_speed:.2f} km/h")

Average of 'vessel_min_speed_kmph': 0.64 km/h
